# LeetCode 3887 — Incremental Even-Weighted Cycle Queries bằng Python

- Link: https://leetcode.com/problems/incremental-even-weighted-cycle-queries/
- Độ khó: **Hard**
- Chủ đề: Union-Find có trọng số chẵn lẻ (parity DSU / weighted DSU), đồ thị hai phía tổng quát, nhân đôi đỉnh, BFS gán nhãn

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def numberOfEdgesAdded(self, n: int, edges: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Chứng minh: "**mọi chu trình có tổng trọng số chẵn**" ⇔ tồn tại cách gán nhãn `p(v) ∈ {0, 1}` sao cho mọi cạnh `(u, v, w)` thỏa `p(u) ⊕ p(v) = w`.
2. Hiểu và cài đặt **DSU chẵn lẻ**: mỗi đỉnh lưu `par[v]` = XOR từ `v` lên cha; `find` trả về gốc và XOR tới gốc, có nén đường đi.
3. Kiểm tra cạnh mới trong `O(α(n))`: nếu hai đầu đã cùng thành phần thì XOR đường đi phải bằng `w`.
4. Biết cách **nhân đôi đỉnh** (`v` và `v + n`) để dùng DSU thường cho cùng bài toán.
5. So sánh với cách BFS gán nhãn lại mỗi lần (`O(m (n + m))`).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Đồ thị vô hướng `n` đỉnh, ban đầu không có cạnh. Xử lý lần lượt các cạnh `[u, v, w]` (`w ∈ {0, 1}`): **chỉ thêm** cạnh nếu sau khi thêm, **mọi chu trình** trong đồ thị có tổng trọng số **chẵn**. Đếm số cạnh được thêm.

**Ví dụ 1:** `n = 3, edges = [[0,1,1],[1,2,1],[0,2,1]]` → `2` (cạnh thứ ba tạo chu trình tổng 3 — lẻ).

**Ví dụ 2:** `n = 3, edges = [[0,1,1],[1,2,1],[0,2,0]]` → `3` (chu trình tổng 2 — chẵn).

**Ràng buộc:** `3 ≤ n ≤ 5·10^4`, `1 ≤ edges.length ≤ 5·10^4`, `0 ≤ u < v < n`, cạnh phân biệt, `w ∈ {0, 1}`.

In [1]:
from typing import List
from collections import deque
import random, time

examples = [(3, [[0, 1, 1], [1, 2, 1], [0, 2, 1]], 2), (3, [[0, 1, 1], [1, 2, 1], [0, 2, 0]], 3)]
for n, e, exp in examples:
    print(n, e, "-> kỳ vọng", exp)

3 [[0, 1, 1], [1, 2, 1], [0, 2, 1]] -> kỳ vọng 2
3 [[0, 1, 1], [1, 2, 1], [0, 2, 0]] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. XOR và chẵn lẻ của tổng

Với số 0/1: tổng chẵn ⇔ XOR bằng 0. XOR có tính giao hoán, kết hợp, `a ⊕ a = 0` ⇒ XOR dọc một đường đi rồi quay lại bằng một đường khác cho XOR của chu trình.

In [2]:
from functools import reduce
ws = [1, 1, 0, 1]
print("tổng:", sum(ws), "| chẵn?", sum(ws) % 2 == 0, "| XOR:", reduce(lambda a, b: a ^ b, ws))

tổng: 3 | chẵn? False | XOR: 1


### 3.2. Union-Find (DSU) cơ bản

`parent[v]`, `find(v)` có nén đường đi, `union(a, b)` gộp theo kích thước. Mỗi thao tác gần như `O(1)` (hàm Ackermann ngược `α(n)`).

In [3]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n)); self.size = [1] * n
    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:
            self.parent[x], x = root, self.parent[x]
        return root
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return False
        if self.size[ra] < self.size[rb]: ra, rb = rb, ra
        self.parent[rb] = ra; self.size[ra] += self.size[rb]
        return True

d = DSU(5); d.union(0, 1); d.union(3, 4)
print([d.find(i) for i in range(5)])

[0, 0, 2, 3, 3]


### 3.3. BFS gán nhãn trên một thành phần

Gán `p(start) = 0`, đi theo cạnh `(u, v, w)`: `p(v) = p(u) ⊕ w`. Nếu gặp đỉnh đã có nhãn mà `p(v) ≠ p(u) ⊕ w` ⇒ có chu trình lẻ.

In [4]:
def consistent(n, edge_list):
    adj = [[] for _ in range(n)]
    for u, v, w in edge_list:
        adj[u].append((v, w)); adj[v].append((u, w))
    p = [-1] * n
    for s in range(n):
        if p[s] != -1: continue
        p[s] = 0; dq = deque([s])
        while dq:
            u = dq.popleft()
            for v, w in adj[u]:
                if p[v] == -1:
                    p[v] = p[u] ^ w; dq.append(v)
                elif p[v] != p[u] ^ w:
                    return False
    return True

print(consistent(3, [[0, 1, 1], [1, 2, 1], [0, 2, 1]]), consistent(3, [[0, 1, 1], [1, 2, 1], [0, 2, 0]]))

False True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

**Bổ đề.** Mọi chu trình có tổng trọng số chẵn ⇔ tồn tại nhãn `p(v) ∈ {0,1}` với `p(u) ⊕ p(v) = w` cho mọi cạnh.

- (⇐) Đi quanh chu trình `v₀ → v₁ → … → v₀`: XOR các trọng số = `⊕ (p(vᵢ) ⊕ p(vᵢ₊₁))` = `0` (mỗi nhãn xuất hiện hai lần).
- (⇒) Lấy cây khung của mỗi thành phần, gán nhãn theo cây. Một cạnh ngoài cây `(u, v, w)` cùng đường đi trên cây tạo chu trình; chu trình chẵn ⇒ `w = p(u) ⊕ p(v)`.

Trực giác: `w = 0` nghĩa là "hai đầu **cùng màu**", `w = 1` nghĩa là "hai đầu **khác màu**". Đây là bài tô hai màu có ràng buộc "cùng/khác".

Tính tay ví dụ 1: `0 -1- 1` ⇒ khác màu; `1 -1- 2` ⇒ khác màu ⇒ `0` và `2` **cùng màu**. Cạnh `0 -1- 2` đòi khác màu ⇒ mâu thuẫn ⇒ bỏ. Ví dụ 2: cạnh `0 -0- 2` đòi cùng màu ⇒ thêm được.

## 5. Cách 1 — Brute Force: thử thêm rồi kiểm tra toàn đồ thị

Với mỗi cạnh: thêm tạm vào danh sách, chạy `consistent` (BFS mọi thành phần). Nếu nhất quán thì giữ, ngược lại bỏ. `O(m · (n + m))`.

In [5]:
def brute(n, edges):
    kept = []
    for e in edges:
        if consistent(n, kept + [e]):
            kept.append(e)
    return len(kept)

for n, e, exp in examples:
    print(brute(n, e), "kỳ vọng", exp)

2 kỳ vọng 2
3 kỳ vọng 3


**Nút thắt:** mỗi cạnh lại BFS **toàn bộ** đồ thị. Thực ra chỉ cần biết: hai đầu có cùng thành phần không, và nếu có thì **XOR đường đi** giữa chúng bằng bao nhiêu.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** Đồ thị đã nhận luôn nhất quán ⇒ trong một thành phần, XOR của **mọi** đường đi giữa `u` và `v` đều bằng `p(u) ⊕ p(v)` (không phụ thuộc đường đi).

**Insight 2 — DSU chẵn lẻ:** lưu `rel[v]` = XOR từ `v` tới `parent[v]`. Khi `find(v)` nén đường đi, cập nhật `rel[v]` = XOR từ `v` thẳng tới gốc. Khi đó `p(u) ⊕ p(v) = rel[u] ⊕ rel[v]` (cùng gốc).

**Quy tắc "nếu … thì …"** với cạnh `(u, v, w)`:

- nếu `find(u) ≠ find(v)`: **gộp** — đặt gốc `rv` làm con của `ru` với `rel[rv] = rel[u] ⊕ rel[v] ⊕ w`; đếm cạnh;
- nếu cùng gốc và `rel[u] ⊕ rel[v] = w`: chu trình mới chẵn ⇒ **thêm** (không cần gộp); đếm;
- nếu cùng gốc và `rel[u] ⊕ rel[v] ≠ w`: chu trình lẻ ⇒ **bỏ**.

Vì sao chỉ cần kiểm tra chu trình **mới**? Theo bổ đề, đồ thị mới nhất quán ⇔ nhãn cũ (duy nhất trên mỗi thành phần, sai khác đảo màu) thỏa thêm cạnh mới.

## 7. Các cách tiếp theo

- **Cách 2:** BFS tìm XOR đường đi giữa `u` và `v` trong đồ thị đã nhận (chỉ duyệt thành phần chứa `u`) — `O(m · (n + m))` tệ nhất nhưng thường nhanh hơn Cách 1.
- **Cách 3:** **DSU chẵn lẻ** — `O(m α(n))`, lời giải nộp chính.
- **Cách 4:** **DSU nhân đôi đỉnh** (`v` = "màu 0", `v + n` = "màu 1") — DSU thường, `O(m α(n))`.

### 7.1. Cách 2 — BFS XOR đường đi

In [6]:
def bfs_path_xor(n, edges):
    adj = [[] for _ in range(n)]
    cnt = 0
    for u, v, w in edges:
        # BFS từ u để tìm nhãn tương đối của v
        lab = {u: 0}
        dq = deque([u])
        while dq and v not in lab:
            x = dq.popleft()
            for y, wy in adj[x]:
                if y not in lab:
                    lab[y] = lab[x] ^ wy
                    dq.append(y)
        if v in lab and lab[v] != w:
            continue                         # chu trình lẻ
        adj[u].append((v, w)); adj[v].append((u, w))
        cnt += 1
    return cnt

for n, e, exp in examples:
    print(bfs_path_xor(n, e), "kỳ vọng", exp)

2 kỳ vọng 2
3 kỳ vọng 3


**Độ phức tạp:** mỗi cạnh BFS một thành phần `O(n + m)` ⇒ tệ nhất `O(m(n + m)) ≈ 5·10^9`. Quá chậm cho ràng buộc.

### 7.2. Cách 3 — DSU chẵn lẻ (lời giải chính)

**Giải thích từng đoạn:**

1. `parent[v] = v`, `rel[v] = 0`, `size[v] = 1`.
2. `find(v)`: đi lên tìm gốc, đồng thời tính XOR; sau đó nén: mọi đỉnh trên đường trỏ thẳng về gốc với `rel` = XOR tới gốc. Viết **lặp** để tránh đệ quy sâu.
3. Xử lý cạnh như mục 6.

In [7]:
class ParityDSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rel = [0] * n                # XOR từ v tới parent[v]
        self.size = [1] * n
    def find(self, v):
        path = []
        while self.parent[v] != v:
            path.append(v)
            v = self.parent[v]
        root = v
        # nén từ đỉnh gần gốc ra xa: rel tích lũy
        acc = 0
        for x in reversed(path):
            acc ^= self.rel[x]
            self.rel[x] = acc
            self.parent[x] = root
        return root
    def add(self, u, v, w):
        ru, rv = self.find(u), self.find(v)
        if ru == rv:
            return (self.rel[u] ^ self.rel[v]) == w
        if self.size[ru] < self.size[rv]:
            ru, rv, u, v = rv, ru, v, u
        self.parent[rv] = ru
        self.rel[rv] = self.rel[u] ^ self.rel[v] ^ w
        self.size[ru] += self.size[rv]
        return True

def parity_dsu(n, edges):
    d = ParityDSU(n)
    return sum(d.add(u, v, w) for u, v, w in edges)

for n, e, exp in examples:
    print(parity_dsu(n, e), "kỳ vọng", exp)

2 kỳ vọng 2
3 kỳ vọng 3


**Lưu ý phần nén:** đường đi `path = [v, cha(v), …]` (từ dưới lên). Duyệt **ngược** (từ đỉnh sát gốc), `acc` là XOR từ đỉnh đó tới gốc; gán `rel[x] = acc`. Do `rel[x]` cũ là XOR tới cha, và `acc` trước khi cộng là XOR từ cha tới gốc, nên `acc ^= rel[x]` cho đúng XOR từ `x` tới gốc.

**Độ phức tạp Cách 3:** `O(m α(n))` thời gian, `O(n)` bộ nhớ.

### 7.3. Cách 4 — DSU nhân đôi đỉnh

Đỉnh `v` ứng với "màu của `v` là 0" và `v + n` ứng với "màu của `v` là 1" (theo nghĩa tương đối). Cạnh `(u, v, w)`:

- `w = 0` (cùng màu): gộp `(u, v)` và `(u+n, v+n)`; **mâu thuẫn** nếu đã có `find(u) == find(v+n)`.
- `w = 1` (khác màu): gộp `(u, v+n)` và `(u+n, v)`; **mâu thuẫn** nếu đã có `find(u) == find(v)`.

In [8]:
def doubled_dsu(n, edges):
    d = DSU(2 * n)
    cnt = 0
    for u, v, w in edges:
        if w == 0:
            if d.find(u) == d.find(v + n):
                continue
            d.union(u, v); d.union(u + n, v + n)
        else:
            if d.find(u) == d.find(v):
                continue
            d.union(u, v + n); d.union(u + n, v)
        cnt += 1
    return cnt

for n, e, exp in examples:
    print(doubled_dsu(n, e), "kỳ vọng", exp)

2 kỳ vọng 2
3 kỳ vọng 3


In [9]:
class SolutionDoubled:
    def numberOfEdgesAdded(self, n: int, edges: List[List[int]]) -> int:
        return doubled_dsu(n, edges)

print([SolutionDoubled().numberOfEdgesAdded(n, e) for n, e, _ in examples])

[2, 3]


**Độ phức tạp Cách 4:** `O(m α(n))`, bộ nhớ `O(2n)`. Đơn giản hơn về mặt khái niệm (không cần `rel`), đổi lại gấp đôi số đỉnh.

## 8. TRACE từng bước

Trace Cách 3 với `n = 5`, cạnh: `[0,1,1], [1,2,0], [3,4,1], [2,3,1], [0,4,0], [0,4,1]` (tự dựng).

| cạnh | gốc(u), gốc(v) | rel[u] ⊕ rel[v] | hành động |
|---|---|---|---|
| 0-1 w=1 | khác | – | gộp, rel[gốc 1] = 1 |
| 1-2 w=0 | khác | – | gộp |
| 3-4 w=1 | khác | – | gộp |
| 2-3 w=1 | khác | – | gộp hai thành phần |
| 0-4 w=0 | cùng | 0⊕? | so sánh |
| 0-4 w=1 | cùng | … | một trong hai cạnh cuối bị bỏ |

Hàm trace in giá trị thực.

In [10]:
def trace_parity(n, edges):
    d = ParityDSU(n); cnt = 0
    for u, v, w in edges:
        ru, rv = d.find(u), d.find(v)
        if ru == rv:
            x = d.rel[u] ^ d.rel[v]
            ok = x == w
            print(f"cạnh {u}-{v} w={w}: cùng gốc {ru}, XOR đường đi = {x} -> {'thêm (chu trình chẵn)' if ok else 'BỎ (chu trình lẻ)'}")
        else:
            ok = d.add(u, v, w)
            print(f"cạnh {u}-{v} w={w}: khác gốc ({ru}, {rv}) -> gộp | parent={d.parent} rel={d.rel}")
        cnt += ok
    print("số cạnh thêm được:", cnt)

trace_parity(5, [[0, 1, 1], [1, 2, 0], [3, 4, 1], [2, 3, 1], [0, 4, 0], [0, 4, 1]])

cạnh 0-1 w=1: khác gốc (0, 1) -> gộp | parent=[0, 0, 2, 3, 4] rel=[0, 1, 0, 0, 0]
cạnh 1-2 w=0: khác gốc (0, 2) -> gộp | parent=[0, 0, 0, 3, 4] rel=[0, 1, 1, 0, 0]
cạnh 3-4 w=1: khác gốc (3, 4) -> gộp | parent=[0, 0, 0, 3, 3] rel=[0, 1, 1, 0, 1]
cạnh 2-3 w=1: khác gốc (0, 3) -> gộp | parent=[0, 0, 0, 0, 3] rel=[0, 1, 1, 0, 1]
cạnh 0-4 w=0: cùng gốc 0, XOR đường đi = 1 -> BỎ (chu trình lẻ)
cạnh 0-4 w=1: cùng gốc 0, XOR đường đi = 1 -> thêm (chu trình chẵn)
số cạnh thêm được: 5


In [11]:
# Đối chiếu nhãn: sau khi xử lý, p(v) = rel[v] (sau find) phải thỏa mọi cạnh ĐÃ NHẬN
n, E = 5, [[0, 1, 1], [1, 2, 0], [3, 4, 1], [2, 3, 1], [0, 4, 0], [0, 4, 1]]
d = ParityDSU(n); kept = [e for e in E if d.add(*e)]
for v in range(n): d.find(v)
print("nhãn:", d.rel, "| cạnh đã nhận thỏa nhãn:", all(d.rel[u] ^ d.rel[v] == w for u, v, w in kept))

nhãn: [0, 1, 1, 0, 1] | cạnh đã nhận thỏa nhãn: True


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def numberOfEdgesAdded(self, n: int, edges: List[List[int]]) -> int:
        parent = list(range(n))
        rel = [0] * n
        size = [1] * n

        def find(v):
            path = []
            while parent[v] != v:
                path.append(v)
                v = parent[v]
            acc = 0
            for x in reversed(path):
                acc ^= rel[x]
                rel[x] = acc
                parent[x] = v
            return v

        cnt = 0
        for u, v, w in edges:
            ru, rv = find(u), find(v)
            if ru == rv:
                if rel[u] ^ rel[v] == w:
                    cnt += 1
                continue
            if size[ru] < size[rv]:
                ru, rv, u, v = rv, ru, v, u
            parent[rv] = ru
            rel[rv] = rel[u] ^ rel[v] ^ w
            size[ru] += size[rv]
            cnt += 1
        return cnt

sol = Solution()
for n, e, exp in examples:
    got = sol.numberOfEdgesAdded(n, e)
    print(got, "✔" if got == exp else "✘")

2 ✔
3 ✔


## 10. Vì sao thuật toán đúng

1. **Bổ đề** (mục 4) đưa điều kiện "mọi chu trình chẵn" về "tồn tại nhãn `p` thỏa mọi cạnh".
2. **Bất biến DSU:** với mọi đỉnh `x`, `rel[x]` = `p(x) ⊕ p(parent[x])` cho một nhãn `p` hợp lệ của đồ thị đã nhận; sau `find(x)`, `rel[x] = p(x) ⊕ p(gốc)`.
   - Nén đường đi giữ bất biến (XOR bắc cầu).
   - Gộp `rv` vào `ru` với `rel[rv] = rel[u] ⊕ rel[v] ⊕ w`: ta đảo (hoặc không) toàn bộ nhãn thành phần của `rv` để `p(u) ⊕ p(v) = w`. Vì hai thành phần rời nhau trước đó, việc đảo màu cả thành phần không phá cạnh nào.
3. **Cùng thành phần:** nhãn trên một thành phần liên thông là **duy nhất sai khác đảo toàn bộ** (nhãn của một đỉnh quyết định mọi đỉnh). Vậy cạnh mới hợp lệ ⇔ `p(u) ⊕ p(v) = w` ⇔ `rel[u] ⊕ rel[v] = w`.
4. Cạnh bị bỏ không ảnh hưởng các bước sau (đồ thị không đổi), đúng như đề.

## 11. Phân tích Time / Space Complexity

Đặt `n` = số đỉnh, `m` = số cạnh.

| Cách | Time | Space |
|---|---|---|
| 1. Thêm thử + BFS toàn đồ thị | `O(m (n + m))` | `O(n + m)` |
| 2. BFS XOR đường đi | `O(m (n + m))` tệ nhất | `O(n + m)` |
| 3. DSU chẵn lẻ | `O(m α(n))` | `O(n)` |
| 4. DSU nhân đôi | `O(m α(n))` | `O(n)` |

## 12. Bộ test

In [13]:
def run_all(n, e, with_brute=True):
    res = [bfs_path_xor(n, e), parity_dsu(n, e), doubled_dsu(n, e), Solution().numberOfEdgesAdded(n, e)]
    if with_brute:
        res.append(brute(n, e))
    return res

for n, e, exp in examples:
    assert all(v == exp for v in run_all(n, e))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [14]:
# 12.2 ca biên (tự dựng)
cases = [
    (3, [[0, 1, 0]], 1),                                      # 1 cạnh
    (3, [[0, 1, 1], [1, 2, 1], [0, 2, 0]], 3),                # tam giác chẵn
    (4, [[0, 1, 1], [1, 2, 1], [2, 3, 1], [0, 3, 1]], 4),     # chu trình 4 cạnh trọng số 1: tổng 4 chẵn
    (4, [[0, 1, 1], [1, 2, 1], [2, 3, 1], [0, 3, 0]], 3),     # tổng 3 lẻ -> bỏ cạnh cuối
    (5, [[0, 1, 0], [1, 2, 0], [0, 2, 1], [0, 2, 0]], 3),     # (cạnh phân biệt theo đề; ở đây thử cả cạnh trùng để minh họa)
]
for n, e, exp in cases:
    r = run_all(n, e)
    assert all(v == exp for v in r), (n, e, r)
print("Ca biên: OK")

Ca biên: OK


In [15]:
# 12.3 phản ví dụ cho ý tưởng sai "chỉ cần đếm số cạnh trọng số 1 trong thành phần chẵn"
def wrong_count_ones(n, edges):
    d = DSU(n); ones = [0] * n; cnt = 0
    for u, v, w in edges:
        ru, rv = d.find(u), d.find(v)
        if ru == rv:
            if (ones[ru] + w) % 2:                # SAI: dùng tổng trọng số cả thành phần
                continue
            ones[ru] += w
        else:
            d.union(u, v); r = d.find(u); ones[r] = ones[ru] + ones[rv] + w
        cnt += 1
    return cnt

rng = random.Random(12)
for _ in range(3000):
    n = rng.randint(3, 6); pairs = [(u, v) for u in range(n) for v in range(u + 1, n)]; rng.shuffle(pairs)
    e = [[u, v, rng.randint(0, 1)] for u, v in pairs]
    if wrong_count_ones(n, e) != Solution().numberOfEdgesAdded(n, e):
        print("phản ví dụ:", n, e, "| sai:", wrong_count_ones(n, e), "| đúng:", Solution().numberOfEdgesAdded(n, e))
        break
# Lý do: chu trình mới chỉ gồm cạnh mới + MỘT đường đi, không phải mọi cạnh của thành phần.

phản ví dụ: 6 [[0, 2, 0], [1, 4, 0], [4, 5, 1], [2, 5, 1], [3, 4, 0], [0, 4, 1], [2, 3, 0], [0, 1, 0], [1, 3, 0], [0, 3, 0], [1, 2, 1], [3, 5, 1], [1, 5, 0], [2, 4, 0], [0, 5, 0]] | sai: 12 | đúng: 11


In [16]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3887)
for _ in range(300):
    n = rng.randint(3, 9)
    pairs = [(u, v) for u in range(n) for v in range(u + 1, n)]
    rng.shuffle(pairs)
    e = [[u, v, rng.randint(0, 1)] for u, v in pairs[:rng.randint(1, len(pairs))]]
    r = run_all(n, e)
    assert len(set(r)) == 1, (n, e, r)
print("300 test ngẫu nhiên: OK")

300 test ngẫu nhiên: OK


In [17]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(5)
n = 5 * 10**4
seen = set(); e = []
while len(e) < 5 * 10**4:
    u, v = sorted(rng.sample(range(n), 2))
    if (u, v) not in seen:
        seen.add((u, v)); e.append([u, v, rng.randint(0, 1)])
t0 = time.time(); a = Solution().numberOfEdgesAdded(n, e); t1 = time.time()
b = doubled_dsu(n, e)
print("đáp án:", a, "| khớp nhân đôi:", a == b, "| thời gian < 1s:", t1 - t0 < 1)
# chuỗi dài: cây đường thẳng rồi các cạnh đóng chu trình
e2 = [[i, i + 1, i % 2] for i in range(n - 1)] + [[0, j, 1] for j in range(2, 500)]
print("chuỗi dài:", Solution().numberOfEdgesAdded(n, e2), "| BFS:", bfs_path_xor(n, e2[:2000]) == Solution().numberOfEdgesAdded(n, e2[:2000]))

đáp án: 45915 | khớp nhân đôi: True | thời gian < 1s: True


chuỗi dài: 50249 | BFS: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên cập nhật `rel` khi nén đường đi

Nếu chỉ đổi `parent[x] = root` mà giữ `rel[x]` cũ (XOR tới cha cũ), các truy vấn sau sẽ sai.

In [18]:
def wrong_no_rel_update(n, edges):
    parent = list(range(n)); rel = [0] * n
    def find(v):
        path = []
        while parent[v] != v:
            path.append(v); v = parent[v]
        for x in path:
            parent[x] = v                         # SAI: không cập nhật rel[x]
        return v
    cnt = 0
    for u, v, w in edges:
        ru, rv = find(u), find(v)
        if ru == rv:
            cnt += (rel[u] ^ rel[v]) == w; continue
        parent[rv] = ru; rel[rv] = rel[u] ^ rel[v] ^ w; cnt += 1
    return cnt

rng = random.Random(1)
for _ in range(3000):
    n = rng.randint(3, 7)
    pairs = [(u, v) for u in range(n) for v in range(u + 1, n)]; rng.shuffle(pairs)
    e = [[u, v, rng.randint(0, 1)] for u, v in pairs]
    if wrong_no_rel_update(n, e) != Solution().numberOfEdgesAdded(n, e):
        print("phản ví dụ:", n, e, "| sai:", wrong_no_rel_update(n, e), "| đúng:", Solution().numberOfEdgesAdded(n, e)); break

phản ví dụ: 7 [[2, 4, 1], [1, 4, 0], [2, 5, 1], [1, 5, 1], [0, 3, 1], [0, 6, 1], [4, 5, 0], [0, 2, 1], [1, 2, 0], [0, 1, 1], [0, 5, 1], [3, 5, 0], [4, 6, 1], [1, 6, 1], [0, 4, 0], [2, 3, 1], [5, 6, 0], [3, 6, 0], [3, 4, 1], [2, 6, 1], [1, 3, 1]] | sai: 9 | đúng: 13


### Lỗi 2 — Gộp sai công thức `rel[rv] = w` (quên XOR với `rel[u]`, `rel[v]`)

`rel[rv]` là XOR từ **gốc** `rv` tới gốc `ru`, không phải từ `v` tới `u`. Phải bù phần `p(u) ⊕ p(ru)` và `p(v) ⊕ p(rv)`.

In [19]:
def wrong_union_formula(n, edges):
    d = ParityDSU(n); cnt = 0
    for u, v, w in edges:
        ru, rv = d.find(u), d.find(v)
        if ru == rv:
            cnt += (d.rel[u] ^ d.rel[v]) == w; continue
        d.parent[rv] = ru; d.rel[rv] = w; cnt += 1          # SAI
    return cnt

e = [[0, 1, 1], [2, 3, 0], [1, 2, 0], [0, 3, 0]]
print("sai:", wrong_union_formula(4, e), "| đúng:", Solution().numberOfEdgesAdded(4, e), "| vét cạn:", brute(4, e))

sai: 4 | đúng: 3 | vét cạn: 3


### Lỗi 3 — Không thêm cạnh "đóng chu trình chẵn" vào kết quả

Khi `u`, `v` cùng thành phần và XOR khớp, cạnh **vẫn được thêm** (đếm), chỉ là DSU không cần gộp. Quên `cnt += 1` ở nhánh này sẽ đếm thiếu.

In [20]:
def wrong_skip_same(n, edges):
    d = ParityDSU(n); cnt = 0
    for u, v, w in edges:
        if d.find(u) == d.find(v):
            continue                                       # SAI: bỏ qua cả trường hợp hợp lệ
        d.add(u, v, w); cnt += 1
    return cnt

print("sai:", wrong_skip_same(3, examples[1][1]), "| đúng:", Solution().numberOfEdgesAdded(3, examples[1][1]))

sai: 2 | đúng: 3


### Lỗi 4 — `find` đệ quy trên chuỗi dài

Trước khi nén, cây DSU có thể sâu (nếu không gộp theo kích thước) ⇒ đệ quy vượt giới hạn 1000. **Sửa:** `find` lặp (như trên) và/hoặc gộp theo kích thước.

In [21]:
import sys
print("giới hạn đệ quy:", sys.getrecursionlimit(), "| n tối đa:", 5 * 10**4)

giới hạn đệ quy: 1000 | n tối đa: 50000


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mọi chu trình có tổng chẵn"**, **"đồ thị hai phía có ràng buộc cùng/khác"**, **"phương trình `x_u ⊕ x_v = w`"** ⇒ DSU chẵn lẻ (hoặc nhân đôi đỉnh).
- **Thêm cạnh online, cạnh mâu thuẫn bị bỏ** ⇒ DSU kiểm tra trước khi gộp.
- DSU có trọng số tổng quát: `rel` có thể là tổng, hiệu, tích modulo… miễn là phép toán tạo thành **nhóm** (có nghịch đảo).
- Bài liên quan: *Is Graph Bipartite?* (LC 785), *Possible Bipartition* (LC 886), *Evaluate Division* (LC 399 — DSU có trọng số tỉ lệ).

## 15. Biến thể của bài

1. **Mọi chu trình có tổng chia hết cho `M`** (trọng số `0..M-1`): DSU với `rel` cộng modulo `M`.
2. **Mọi chu trình có tổng = 0** với trọng số có dấu và cạnh có hướng (`p(v) - p(u) = w`): DSU tiềm năng (potential).
3. **Trả về danh sách cạnh bị bỏ** hoặc thời điểm đầu tiên có chu trình lẻ.
4. **Xóa cạnh** (fully dynamic): cần cấu trúc mạnh hơn (DSU rollback offline + chia để trị theo thời gian).

In [22]:
# Biến thể 1: tổng mọi chu trình chia hết cho M (trọng số 0..M-1, chu trình đi theo một chiều với dấu)
def mod_cycles(n, edges, M):
    parent = list(range(n)); rel = [0] * n            # rel[v] = p(v) - p(parent[v]) (mod M)
    def find(v):
        path = []
        while parent[v] != v:
            path.append(v); v = parent[v]
        acc = 0
        for x in reversed(path):
            acc = (acc + rel[x]) % M; rel[x] = acc; parent[x] = v
        return v
    cnt = 0
    for u, v, w in edges:                              # ràng buộc p(v) - p(u) = w (mod M)
        ru, rv = find(u), find(v)
        if ru == rv:
            cnt += (rel[v] - rel[u]) % M == w % M; continue
        parent[rv] = ru; rel[rv] = (rel[u] + w - rel[v]) % M; cnt += 1
    return cnt

# với M = 2 và đồ thị vô hướng, trùng với bài gốc
rng = random.Random(15)
for _ in range(200):
    n = rng.randint(3, 8); pairs = [(u, v) for u in range(n) for v in range(u + 1, n)]; rng.shuffle(pairs)
    e = [[u, v, rng.randint(0, 1)] for u, v in pairs]
    assert mod_cycles(n, e, 2) == Solution().numberOfEdgesAdded(n, e)
print("Biến thể 1 với M = 2 khớp lời giải gốc.")

Biến thể 1 với M = 2 khớp lời giải gốc.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thêm thử + BFS | Kiểm tra lại toàn đồ thị | `O(m(n+m))` | `O(n+m)` | Kiểm thử |
| 2. BFS XOR đường đi | Chỉ duyệt thành phần của `u` | `O(m(n+m))` | `O(n+m)` | `m` nhỏ |
| 3. DSU chẵn lẻ | `rel` = XOR tới gốc | `O(m α(n))` | `O(n)` | **Lời giải chuẩn** |
| 4. DSU nhân đôi | `v`, `v+n` là hai màu | `O(m α(n))` | `O(n)` | Khi chỉ có DSU thường sẵn có |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `n = 4`, `edges = [[0,1,0],[1,2,0],[2,3,0],[0,3,1]]` → ?

**Bài 2 (tính tay).** `n = 4`, `edges = [[0,1,1],[2,3,1],[0,2,0],[1,3,1],[1,3,0]]` (giả sử cho phép cạnh lặp để luyện tập) → từng cạnh nhận hay bỏ?

**Bài 3 (điền TODO).** Viết `is_bipartite_after_each(n, edges)` trả về danh sách bool: sau mỗi cạnh (luôn thêm, mọi `w = 1`), đồ thị còn hai phía không.

**Bài 4 (tự viết lại).** Viết lại Cách 3 với `find` **đệ quy** nhưng an toàn bằng gộp theo kích thước (độ sâu `O(log n)`).

**Bài 5 (mini challenge).** Trả về **chỉ số cạnh đầu tiên bị bỏ** (hoặc `-1`), và nhãn `p` của mọi đỉnh sau khi xử lý.

In [23]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def is_bipartite_after_each(n, edges):
    # TODO: DSU chẵn lẻ với w = 1 cho mọi cạnh; một khi đã mâu thuẫn thì các bước sau đều False
    pass

In [24]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def first_rejected_and_labels(n, edges):
    # TODO: chạy DSU chẵn lẻ; ghi lại chỉ số đầu tiên bị bỏ; cuối cùng find(v) mọi v, p(v) = rel[v]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ba cạnh `w = 0` ⇒ 0, 1, 2, 3 cùng màu. (2) Cạnh `0-3 w=1` đòi khác màu. (3) Bỏ ⇒ `3`.

**Bài 2.** (1) `0≠1`, `2≠3`, `0=2` ⇒ `1 ≠ 2`… suy ra `1 = 3`? (`1 ≠ 0 = 2 ≠ 3` ⇒ `1 = 3`). (2) Cạnh `1-3 w=1` đòi khác ⇒ bỏ. (3) Cạnh `1-3 w=0` ⇒ nhận ⇒ tổng `4`.

**Bài 3.** (1) Mọi cạnh có `w = 1`. (2) Giữ cờ `ok`. (3) Đừng thêm cạnh mâu thuẫn vào DSU (nhưng cờ vẫn `False`).

**Bài 4.** (1) `find` đệ quy trả về `(gốc, xor)`. (2) Gộp gốc nhỏ vào gốc lớn. (3) Độ sâu cây ≤ `log₂ n` ≈ 16.

**Bài 5.** (1) Biến `first = -1`. (2) Gán `first = i` lần đầu `add` trả `False`. (3) Nhãn: `find(v)` rồi lấy `rel[v]`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
assert Solution().numberOfEdgesAdded(4, [[0, 1, 0], [1, 2, 0], [2, 3, 0], [0, 3, 1]]) == 3
assert Solution().numberOfEdgesAdded(4, [[0, 1, 1], [2, 3, 1], [0, 2, 0], [1, 3, 1], [1, 3, 0]]) == 4
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [26]:
# Bài 3
def is_bipartite_after_each_sol(n, edges):
    d = ParityDSU(n); ok = True; out = []
    for u, v in edges:
        if ok and not d.add(u, v, 1):
            ok = False
        out.append(ok)
    return out

assert is_bipartite_after_each_sol(3, [[0, 1], [1, 2], [0, 2]]) == [True, True, False]
assert is_bipartite_after_each_sol(4, [[0, 1], [1, 2], [2, 3], [0, 3]]) == [True] * 4
print("Bài 3 OK")

Bài 3 OK


In [27]:
# Bài 4
def sol_recursive(n, edges):
    parent = list(range(n)); rel = [0] * n; size = [1] * n
    def find(v):
        if parent[v] == v:
            return v
        r = find(parent[v])
        rel[v] ^= rel[parent[v]] if parent[v] != r else 0
        parent[v] = r
        return r
    cnt = 0
    for u, v, w in edges:
        ru, rv = find(u), find(v)
        if ru == rv:
            cnt += (rel[u] ^ rel[v]) == w; continue
        if size[ru] < size[rv]: ru, rv, u, v = rv, ru, v, u
        parent[rv] = ru; rel[rv] = rel[u] ^ rel[v] ^ w; size[ru] += size[rv]; cnt += 1
    return cnt

rng = random.Random(4)
for _ in range(300):
    n = rng.randint(3, 12); pairs = [(u, v) for u in range(n) for v in range(u + 1, n)]; rng.shuffle(pairs)
    e = [[u, v, rng.randint(0, 1)] for u, v in pairs]
    assert sol_recursive(n, e) == Solution().numberOfEdgesAdded(n, e)
print("Bài 4 OK")

Bài 4 OK


In [28]:
# Bài 5
def first_rejected_and_labels_sol(n, edges):
    d = ParityDSU(n); first = -1
    for i, (u, v, w) in enumerate(edges):
        if not d.add(u, v, w) and first == -1:
            first = i
    for v in range(n): d.find(v)
    return first, d.rel[:]

first, lab = first_rejected_and_labels_sol(3, examples[0][1])
assert first == 2
kept = [e for i, e in enumerate(examples[0][1]) if i != 2]
assert all(lab[u] ^ lab[v] == w for u, v, w in kept)
print("Bài 5 OK:", first, lab)

Bài 5 OK: 2 [0, 1, 0]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
parent = list(range(n)); rel = [0]*n; size = [1]*n
def find(v):                      # lặp + nén, rel[v] = XOR tới gốc
    path = []
    while parent[v] != v: path.append(v); v = parent[v]
    acc = 0
    for x in reversed(path): acc ^= rel[x]; rel[x] = acc; parent[x] = v
    return v
for u, v, w in edges:
    ru, rv = find(u), find(v)
    if ru == rv: cnt += (rel[u] ^ rel[v]) == w; continue
    (gộp nhỏ vào lớn) parent[rv] = ru; rel[rv] = rel[u] ^ rel[v] ^ w; cnt += 1
```

**Checklist:**

- [ ] `rel` được cập nhật khi nén đường đi (XOR tích lũy từ gốc xuống)?
- [ ] Công thức gộp `rel[rv] = rel[u] ^ rel[v] ^ w` (sau khi đã `find`)?
- [ ] Cạnh cùng thành phần và khớp XOR vẫn **được đếm**?
- [ ] `find` không đệ quy sâu?
- [ ] Đổi chỗ `u, v` cùng với `ru, rv` khi gộp theo kích thước?